# Index · Unit 8 — Cost, latency & efficiency

Same shape as Units 1–7. For every component in **Section 9 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python /
  scripted *fake model* (no API key needed).
- **SOTA** — how it's actually done today (prompt caching, effort control, streaming, batch API).
  Some need `OPENAI_API_KEY`.

Each component lists **📚 Resources** (also consolidated in `RESOURCES.md`).

Components: **1** prompt caching · **2** token budgets / effort control ·
**3** streaming · **4** batching.

**Intuition for the whole unit:** every token costs money and time, and a slow,
expensive agent is a bad agent no matter how smart. So you **reuse** work that
doesn't change (caching), **cap** what any one task may spend (budgets/effort),
hide latency by **showing partial output early** (streaming), and **amortize**
per-call overhead across many inputs (batching). Same pattern: thin wrappers
around the model, which SOTA features package.

## Setup

In [ ]:
import os, re, json

MODEL = "gpt-4o-mini"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

## 1. Prompt caching

Most calls share a big **stable prefix** — the same long system prompt, tools,
few-shot examples — and change only a little at the end. Recomputing the prefix
every time is wasted money and latency. **Prompt caching** stores the processed
prefix (its KV state) so repeat calls pay for the *suffix* only. Rule of thumb:
**put stable content first, variable content last.**

**From scratch:** a prefix cache that charges full price on a miss, suffix-only on
a hit.

> 📚 **Resources:** [OpenAI — Prompt caching](https://platform.openai.com/docs/guides/prompt-caching) ·
> [Anthropic — Prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)

In [ ]:
# From scratch — cache the stable prefix; pay only for the changing suffix.
def ntok(s): return len(s.split())
_prefix_cache = {}

def call_with_cache(prefix, suffix):
    cost = 0
    if prefix in _prefix_cache:
        print(f"  prefix HIT  (prefix {ntok(prefix)} tok reused, $0)")
    else:
        _prefix_cache[prefix] = True
        cost += ntok(prefix)
        print(f"  prefix MISS (prefix {ntok(prefix)} tok computed)")
    cost += ntok(suffix)                       # suffix always costs
    return cost

SYS = ("You are a careful assistant. Follow all rules. " * 8).strip()   # big stable prefix
print("call 1 cost:", call_with_cache(SYS, "Question: what is 2+2?"))
print("call 2 cost:", call_with_cache(SYS, "Question: capital of France?"))  # prefix cached

In [ ]:
# SOTA — caching is automatic for long prefixes; you just ORDER the prompt right.
# OpenAI caches prompts >~1024 tokens automatically; usage reports cached tokens.
def cached_structure(user_q):
    STABLE = "You are an expert assistant. [imagine a long system prompt + few-shot here]"
    r_msg = chat([{"role": "system", "content": STABLE},   # <- stable FIRST (cacheable)
                  {"role": "user",   "content": user_q}])   # <- variable LAST
    # In the raw response, usage.prompt_tokens_details.cached_tokens shows the hit.
    return r_msg.content

print(cached_structure("Say 'cached' in one word."))

## 2. Token budgets / effort control

Left unchecked, an agent can loop and burn unbounded tokens. A **budget** caps
spend per task: count tokens, refuse/trim when the cap is hit. The newer knob is
**effort control** — telling a reasoning model *how hard to think* (low effort =
cheaper/faster, high = more thorough) so you match compute to task difficulty.

**From scratch:** a budget tracker that trims context to fit and stops when
exhausted.

> 📚 **Resources:** [OpenAI — reasoning effort](https://platform.openai.com/docs/guides/reasoning) ·
> [OpenAI — managing tokens](https://platform.openai.com/docs/advanced-usage)

In [ ]:
# From scratch — a token budget that trims context and halts when spent.
def ntok(s): return max(1, len(s) // 4)          # ~4 chars/token

class Budget:
    def __init__(self, limit): self.limit, self.used = limit, 0
    def charge(self, text):
        t = ntok(text)
        if self.used + t > self.limit:
            raise RuntimeError(f"budget exceeded ({self.used}+{t} > {self.limit})")
        self.used += t
        return t

def fit_to_budget(messages, limit):
    # keep the most recent messages that fit (drop oldest first).
    kept, used = [], 0
    for m in reversed(messages):
        t = ntok(m["content"])
        if used + t > limit: break
        kept.insert(0, m); used += t
    return kept

b = Budget(limit=20)
for step in range(10):
    try: print(f"step {step}: charged {b.charge('some tool output here')}, used={b.used}")
    except RuntimeError as e: print("✋", e); break

msgs = [{"role":"user","content":"old "*20},{"role":"user","content":"mid "*10},{"role":"user","content":"recent"}]
print("fit:", [m["content"][:12] for m in fit_to_budget(msgs, 15)])

In [ ]:
# SOTA — hard cap output tokens, and dial reasoning effort to the task.
def bounded_call(prompt, max_tokens=50):
    return chat([{"role": "user", "content": prompt}],
                max_completion_tokens=max_tokens).content     # hard ceiling on output
    # On reasoning models you'd also pass reasoning_effort="low"|"medium"|"high"
    # (and GPT-5/o-series expose it) to trade cost/latency for depth.

print(bounded_call("List three agent cost-saving tricks, very briefly."))

## 3. Streaming

A model that takes 8s to answer *feels* broken if nothing appears for 8s.
**Streaming** emits tokens as they're generated, so the user sees output almost
immediately (low **time-to-first-token**). The total time is the same — the
*perceived* latency drops dramatically, and you can start downstream work on
partial output.

**From scratch:** a generator that yields chunks; the consumer prints
incrementally.

> 📚 **Resources:** [OpenAI — Streaming responses](https://platform.openai.com/docs/guides/streaming-responses)

In [ ]:
# From scratch — streaming is just a generator of chunks.
import sys, time

def fake_stream(text):
    for word in text.split():
        yield word + " "                       # each yield = one streamed chunk

print("streamed: ", end="")
for chunk in fake_stream("tokens appear one at a time as they are produced"):
    sys.stdout.write(chunk); sys.stdout.flush()
    # time.sleep(0.02)   # uncomment to SEE the effect
print()

In [ ]:
# SOTA — stream=True yields deltas you print as they arrive.
def stream_chat(prompt):
    from openai import OpenAI
    stream = OpenAI().chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], stream=True)
    out = []
    for event in stream:
        delta = event.choices[0].delta.content or ""
        print(delta, end="", flush=True)       # render live
        out.append(delta)
    print()
    return "".join(out)

_ = stream_chat("Count from 1 to 5.")

## 4. Batching

Per-call overhead (network round-trip, auth, setup) dominates when you have many
small requests. **Batching** sends many inputs together so that overhead is paid
once. Two flavours: **sync batching** (one request with a list of inputs, e.g.
embeddings) and the **async Batch API** (submit a big job, get results later, ~50%
cheaper).

**From scratch:** one batched call vs N single calls, counting round-trips.

> 📚 **Resources:** [OpenAI — Batch API](https://platform.openai.com/docs/guides/batch) ·
> [OpenAI — Embeddings (list input)](https://platform.openai.com/docs/guides/embeddings)

In [ ]:
# From scratch — amortize per-call overhead by batching.
roundtrips = {"n": 0}
def embed_one(x):
    roundtrips["n"] += 1                        # each call = one network round-trip
    return len(x)

def embed_batch(xs):
    roundtrips["n"] += 1                        # ONE round-trip for the whole list
    return [len(x) for x in xs]

inputs = ["alpha", "beta", "gamma", "delta"]

roundtrips["n"] = 0
_ = [embed_one(x) for x in inputs];   print("one-by-one round-trips:", roundtrips["n"])
roundtrips["n"] = 0
_ = embed_batch(inputs);              print("batched    round-trips:", roundtrips["n"])

In [ ]:
# SOTA — embeddings take a LIST in one request; the Batch API runs big jobs ~50% off.
def embed_many(texts):
    from openai import OpenAI
    r = OpenAI().embeddings.create(model="text-embedding-3-small", input=texts)  # one call, many vectors
    return [len(d.embedding) for d in r.data]
    # For thousands of requests, upload a .jsonl to the Batch API (async, cheaper):
    #   OpenAI().batches.create(input_file_id=..., endpoint="/v1/chat/completions", completion_window="24h")

print("vector dims per input:", embed_many(["alpha", "beta", "gamma"]))

## Recap

Efficiency is engineering the token stream:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| prompt caching | cache stable prefix, pay for suffix only | automatic prefix caching (order prompt: stable first) |
| token budgets / effort | budget tracker + context trimming | `max_completion_tokens`, `reasoning_effort` |
| streaming | a generator of chunks | `stream=True`, print deltas live |
| batching | one call over a list vs N calls | list-input embeddings, async Batch API (~50% off) |

These stack with everything prior: caching makes long system prompts (Unit 5
guardrails, Unit 4 roles) cheap to reuse; budgets bound the agent loop (Unit 1);
streaming + batching shape how results reach users and downstream agents. The
through-line holds — a few lines of plain Python, packaged by SOTA features.

Next in `index.md`: **Unit 9 — Learning & adaptation** (Section 10) — the final unit.